# Task 1 — Dataset Understanding & Exploratory Analysis
**COMP41840 AI for Health**  
**Owner:** Sergio  
**Dataset:** Multi-Modal Breast Cancer Dataset (780 patients)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

sns.set_theme(style='whitegrid')

DATA_ROOT = Path('../data/raw')
CLASSES = ['benign', 'malignant', 'normal']

## 1.1 — Image Dataset Structure

In [ ]:
# Count images per class
counts = {}
for cls in CLASSES:
    img_dir = DATA_ROOT / cls / 'images'
    counts[cls] = len(list(img_dir.glob('*.png')))
    print(f'{cls}: {counts[cls]} images')

# Bar chart — class distribution
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(counts.keys(), counts.values(), color=['#4C72B0', '#DD8452', '#55A868'])
ax.set_title('Class Distribution (Images)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../results/figures/class_distribution.png', dpi=150)
plt.show()

## 1.2 — Image Resolution & Format

In [ ]:
# Sample image sizes across classes
widths, heights, class_labels = [], [], []

for cls in CLASSES:
    img_dir = DATA_ROOT / cls / 'images'
    paths = list(img_dir.glob('*.png'))[:50]  # sample 50 per class
    for p in paths:
        with Image.open(p) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
            class_labels.append(cls)

size_df = pd.DataFrame({'width': widths, 'height': heights, 'class': class_labels})
print(size_df.groupby('class')[['width', 'height']].describe())

## 1.3 — Sample Images per Class

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
fig.suptitle('Sample Ultrasound Images per Class', fontsize=14)

for row, cls in enumerate(CLASSES):
    img_dir = DATA_ROOT / cls / 'images'
    samples = sorted(img_dir.glob('*.png'))[:3]
    for col, p in enumerate(samples):
        axes[row][col].imshow(Image.open(p), cmap='gray')
        axes[row][col].set_title(f'{cls} — {p.name}', fontsize=8)
        axes[row][col].axis('off')

plt.tight_layout()
plt.savefig('../results/figures/sample_images.png', dpi=150)
plt.show()

## 1.4 — Mask Overlay Visualisation

In [ ]:
# TODO: overlay masks on images to verify mask alignment
# Use malignant/images/MB-3007.png and malignant/masks/MB-3007.png as example
pass

## 1.5 — Tabular / Clinical Data

In [ ]:
# Load clinical CSV
clinical_path = DATA_ROOT.parent / 'clinical.csv'
df = pd.read_csv(clinical_path)

print('Shape:', df.shape)
print('\nColumn types:')
print(df.dtypes)
df.head()

In [ ]:
# Missing value analysis
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
print(pd.concat([missing, missing_pct], axis=1, keys=['count', '%']))

In [ ]:
# Class balance in tabular data
print(df['label'].value_counts())
df['label'].value_counts().plot(kind='bar', title='Tabular Label Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of numerical features
numerical_cols = df.select_dtypes(include='number').columns.tolist()
df[numerical_cols].hist(figsize=(14, 10), bins=20)
plt.suptitle('Numerical Feature Distributions')
plt.tight_layout()
plt.savefig('../results/figures/feature_distributions.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(df[numerical_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../results/figures/correlation_heatmap.png', dpi=150)
plt.show()

## 1.6 — Summary

| Item | Value |
|------|-------|
| Total patients | 780 |
| Classes | benign, malignant, normal |
| Image modality | Ultrasound (grayscale PNG) |
| Masks available | Yes |
| Tabular features | TBD after loading |
| Missing values | TBD |
| Class imbalance | benign >> malignant > normal |

> **Key finding:** The dataset is imbalanced — benign class dominates. Tasks 2–4 should use weighted loss / `scale_pos_weight` to compensate.